### Pandapower with UK Power Networks - state estimation and forecasting

This tutorial complements the tutorial presented [here](https://github.com/e2nIEE/pandapower/blob/develop/tutorials/ukpn_pp_power_flow.ipynb), which shows how to leverage pandapower for performing studies and analyses on the real grids of UK Power Networks. 
This tutorial shows how to estimate the operating conditions of the grid using ad hoc state estimation techniques able to process measurement or forecast data. 

One of the main challenges in distribution systems is to derive the operating conditions of the grid when only a very limited number of measurements is available. 
To this purpose, this tutorial will show the functionalities of the so-called AF-WLS (Allocation Factor based Weighted Least Squares) method available in pandapower. 
This was conceived specifically to deal with scenarios with very few measurements (so-called *unobservable* grids). 
More details about the technical concepts and the mathematical background behind the AF-WLS state estimation algorithm can be found in the publication available at this [link](https://ieeexplore.ieee.org/abstract/document/10497141).

This tutorial has been created in collaboration with UK Power Networks, the Distribution System Operator owning and operating the electricity network across London, the South East and the East of England.

The tutorial will use the real grids associated with the three licensed electricity distribution networks operated by UK Power Networks (LPN, SPN and EPN).
It will show how to use the AF-WLS state estimation algorithm and its potential performance on reduced portions of the UK Power Networks obtained via the grid reduction algorithm presented in the tutorial available [here](https://github.com/e2nIEE/pandapower/blob/develop/tutorials/ukpn_pp_sensitivity_reduction.ipynb).

UK Power Networks has provided the grid data as part of their LTDS CIM dataset release. It is a "Shared" dataset that requires special access. To request access, visit the [LTDS CIM](https://ukpowernetworks.opendatasoft.com/explore/dataset/ukpn-ltds-cim/information/) page and complete the [Shared Data Request Form](https://ukpowernetworks.opendatasoft.com/login/?next=/explore/forms/cim-access-request-form/). Once approved, CIM data is published as XML file attachments (one per licence area: EPN, SPN, LPN). You can download the XML files directly from the portal.

The additional data required to integrate load and generation in the grid are openly available as Excel tables at the following links: 
- EPN --> [EPN Long Term Development Statement - November 2025](https://ukpowernetworks.sharepoint.com/sites/OpenDataPortalLibrary/Shared%20Documents/Forms/AllItems.aspx?id=%2Fsites%2FOpenDataPortalLibrary%2FShared%20Documents%2FGeneral%2FLong%20Term%20Development%20Statement%2FNovember%202025%2FEPN%20Long%20Term%20Development%20Statement%20%2D%20November%202025&p=true&ga=1)
- SPN --> [SPN Long Term Development Statement - November 2025](https://ukpowernetworks.sharepoint.com/sites/OpenDataPortalLibrary/Shared%20Documents/Forms/AllItems.aspx?id=%2Fsites%2FOpenDataPortalLibrary%2FShared%20Documents%2FGeneral%2FLong%20Term%20Development%20Statement%2FNovember%202025%2FSPN%20Long%20Term%20Development%20Statement%20%2D%20November%202025&p=true&ga=1)
- LPN --> [LPN Long Term Development Statement - November 2025](https://ukpowernetworks.sharepoint.com/sites/OpenDataPortalLibrary/Shared%20Documents/Forms/AllItems.aspx?id=%2Fsites%2FOpenDataPortalLibrary%2FShared%20Documents%2FGeneral%2FLong%20Term%20Development%20Statement%2FNovember%202025%2FLPN%20Long%20Term%20Development%20Statement%20%2D%20November%202025&p=true&ga=1)


In [ ]:
# Import the needed libraries 
import pandapower as pp
import pandapower.topology as top
from pandapower.toolbox import create_replacement_switch_for_branch, select_subnet
from sensitivity_functions import build_reduced_network_from_sensitivity
from pandapower.estimation import state_estimation as se
from pandapower.estimation.util import remove_shunt_injection_from_meas

import pandas as pd
import numpy as np
import copy
import os
pd.options.display.float_format = '{:,.4f}'.format

import warnings
warnings.filterwarnings("ignore")

#### Workarounds for power flow execution
The following blocks of code provide some functions to apply some workarounds necessary to run successfully the power flow on the UK Power Networks grids.
These workarounds include, for example, the creation of external grids (*slack buses* in the power flow terminology) or the replacement of zero impedance components with switches. 

In [ ]:
# Function to replace components with very small impedance with switches.
from pandapower.toolbox import create_replacement_switch_for_branch

def _replace_zero_impedance_components(net):
    min_ohm = 0.001
    to_replace = (np.abs(net.line.x_ohm_per_km * net.line.length_km) <= min_ohm) & net.line.in_service

    if np.any(to_replace):
        print(f"replaced {sum(to_replace)} lines with switches")

    for i in net.line.loc[to_replace].index.values:
        create_replacement_switch_for_branch(net, "line", i)
        net.line.at[i, "in_service"] = False

    xward = net.xward.loc[(np.abs(net.xward.x_ohm) <= min_ohm) & net.xward.in_service].index.values
    if len(xward) > 0:
        pp.replace_xward_by_ward(net, index=xward, drop=False)
        print(f"replaced {len(xward)} xwards with wards")

    zb_f_ohm = np.square(net.bus.loc[net.impedance.from_bus.values, "vn_kv"].values) / net.impedance.sn_mva
    zb_t_ohm = np.square(net.bus.loc[net.impedance.to_bus.values, "vn_kv"].values) / net.impedance.sn_mva
    impedance = ((np.abs(net.impedance.xft_pu) <= min_ohm / zb_f_ohm) |
                (np.abs(net.impedance.xtf_pu) <= min_ohm / zb_t_ohm)) & net.impedance.in_service

    if any(impedance):
        print(f"replaced {sum(impedance)} impedance elements with switches")

    for i in net.impedance.loc[impedance].index.values:
        pp.create_replacement_switch_for_branch(net, "impedance", i)
        net.impedance.at[i, "in_service"] = False

In [ ]:
# Function to apply the needed workarounds
def apply_workarounds(net, license_area, remove_impedance):
    if remove_impedance:
        net.impedance.drop(net.impedance.index, inplace=True)
    _replace_zero_impedance_components(net)
    net.line["c_nf_per_km"] *= 0.1
    net.load["p_mw"] *= 0.1

    if license_area == "LPN":
        pp.create_ext_grid(net,bus=10711,vm_pu=1)
        pp.create_ext_grid(net,bus=10699,vm_pu=1)
        pp.create_ext_grid(net,bus=10674,vm_pu=1)
        pp.create_ext_grid(net,bus=10738,vm_pu=1)
        pp.create_ext_grid(net,bus=10673,vm_pu=1)
    elif license_area == "SPN":
        net.trafo.drop(661,inplace=True)
        pp.create_ext_grid(net,bus=4899,vm_pu=1)
        pp.create_ext_grid(net,bus=4879,vm_pu=1)
        pp.create_ext_grid(net,bus=4903,vm_pu=1)
        pp.create_ext_grid(net,bus=4920,vm_pu=1)
        pp.create_ext_grid(net,bus=4916,vm_pu=1)
        pp.create_ext_grid(net,bus=4925,vm_pu=1)
        pp.create_ext_grid(net,bus=4878,vm_pu=1)
    elif license_area == "EPN":
        pp.create_ext_grid(net,bus=9906,vm_pu=1)
        pp.create_ext_grid(net,bus=9918,vm_pu=1)
        pp.create_ext_grid(net,bus=9900,vm_pu=1)
        pp.create_ext_grid(net,bus=9910,vm_pu=1)
        pp.create_ext_grid(net,bus=9878,vm_pu=1)
    else:
        raise ValueError("Sorry, this license area does not exist in UK Power Networks. Allowed areas are LPN, SPN and EPN.")

    return net

#### Sensitivity-based grid reduction
The following block implements the functions necessary to carry out the grid reduction based on sensitivity factors. 

The **goal** of the grid reduction is to reduce the grid around a user-selected bus of interest while keeping, inside the reduced grid, the same power flow behaviour as in the original-size grid. 

The main **criterion** for the grid reduction is to cut the grid at transformer level based on the sensitivity of the transformers to the changes applied at the bus of interest. In this way, only the portion of the grid directly affected by changes at the bus of interest is kept within the reduced grid model, whereas other parts of the grid that are not influenced by power variations at the bus of interest are excluded from the model and replaced with equivalent elements. 

This grid reduction process allows therefore to create reduced grid models around a selected bus and to focus the analysis on a smaller (and hence more easily manageable) portion of the UK Power Networks grid. 


In [ ]:
def calc_trafo_current_sensitivity_from_power_flow(net_start, net_post, min_i_ka=1e-6):
    """
    Function to compute the sensitivity of transformers to a power change at the bus of interest
    """
    rows = []
    for tidx, tr in net_start.trafo[net_start.trafo.in_service].iterrows():
        tidx = int(tidx)

        hv = int(tr.hv_bus)
        lv = int(tr.lv_bus)

        # initial currents from PF
        i0_hv_ka_start = abs(float(net_start.res_trafo.i_hv_ka.loc[tidx]))
        i0_lv_ka_start = abs(float(net_start.res_trafo.i_lv_ka.loc[tidx]))

        # currents after perturbation from PF
        i0_hv_ka_post = abs(float(net_post.res_trafo.i_hv_ka.loc[tidx]))
        i0_lv_ka_post = abs(float(net_post.res_trafo.i_lv_ka.loc[tidx]))

        # current difference between before and after perturbation
        dI_hv_ka = i0_hv_ka_start - i0_hv_ka_post
        dI_lv_ka = i0_lv_ka_start - i0_lv_ka_post

        # sensitivity computation
        sf_hv = dI_hv_ka / max(i0_hv_ka_start, float(min_i_ka)) if np.isfinite(dI_hv_ka) else np.nan
        sf_lv = dI_lv_ka / max(i0_lv_ka_start, float(min_i_ka)) if np.isfinite(dI_lv_ka) else np.nan

        rows.append({
            "trafo_index": tidx,
            "hv_bus": hv,
            "lv_bus": lv,
            "vn_hv_kv": float(net_start.bus.vn_kv.loc[hv]),
            "vn_lv_kv": float(net_start.bus.vn_kv.loc[lv]),
            "i0_hv_ka": i0_hv_ka_start,
            "i0_lv_ka": i0_lv_ka_start,
            "i0_max_ka": max(i0_hv_ka_start, i0_lv_ka_start),
            "dI_hv_ka": float(dI_hv_ka) if np.isfinite(dI_hv_ka) else np.nan,
            "dI_lv_ka": float(dI_lv_ka) if np.isfinite(dI_lv_ka) else np.nan,
            "dI_max_ka": max(dI_hv_ka, dI_lv_ka),
            "sf_hv": float(sf_hv) if np.isfinite(sf_hv) else np.nan,
            "sf_lv": float(sf_lv) if np.isfinite(sf_lv) else np.nan,
            "sf_max": float(abs(np.nanmax([sf_hv, sf_lv]))),
        })

    return pd.DataFrame(rows).set_index("trafo_index")


def calc_trafo3w_current_sensitivity_from_power_flow(net_start, net_post, min_i_ka=1e-6):
    """
    Function to compute the sensitivity of 3-winding transformers to a power change at the bus of interest
    """

    rows = []
    for tidx, tr in net_start.trafo3w[net_start.trafo3w.in_service].iterrows():
        tidx = int(tidx)

        hv = int(tr.hv_bus)
        mv = int(tr.mv_bus)
        lv = int(tr.lv_bus)

        # initial currents from PF
        i0_hv_ka_start = abs(float(net_start.res_trafo3w.i_hv_ka.loc[tidx])) if "i_hv_ka" in net_start.res_trafo3w.columns else np.nan
        i0_mv_ka_start = abs(float(net_start.res_trafo3w.i_mv_ka.loc[tidx])) if "i_mv_ka" in net_start.res_trafo3w.columns else np.nan
        i0_lv_ka_start = abs(float(net_start.res_trafo3w.i_lv_ka.loc[tidx])) if "i_lv_ka" in net_start.res_trafo3w.columns else np.nan

        # currents after perturbation from PF
        i0_hv_ka_post = abs(float(net_post.res_trafo3w.i_hv_ka.loc[tidx])) if "i_hv_ka" in net_post.res_trafo3w.columns else np.nan
        i0_mv_ka_post = abs(float(net_post.res_trafo3w.i_mv_ka.loc[tidx])) if "i_mv_ka" in net_post.res_trafo3w.columns else np.nan
        i0_lv_ka_post = abs(float(net_post.res_trafo3w.i_lv_ka.loc[tidx])) if "i_lv_ka" in net_post.res_trafo3w.columns else np.nan

        # current difference between before and after perturbation
        dI_hv_ka = i0_hv_ka_start - i0_hv_ka_post
        dI_mv_ka = i0_mv_ka_start - i0_mv_ka_post
        dI_lv_ka = i0_lv_ka_start - i0_lv_ka_post

        # sensitivity computation
        sf_hv = dI_hv_ka / max(i0_hv_ka_start, float(min_i_ka)) if np.isfinite(dI_hv_ka) else np.nan
        sf_mv = dI_mv_ka / max(i0_mv_ka_start, float(min_i_ka)) if np.isfinite(dI_mv_ka) else np.nan
        sf_lv = dI_lv_ka / max(i0_lv_ka_start, float(min_i_ka)) if np.isfinite(dI_lv_ka) else np.nan

        rows.append({
            "trafo3w_index": tidx,
            "hv_bus": hv,
            "mv_bus": mv,
            "lv_bus": lv,
            "vn_hv_kv": float(net_start.bus.vn_kv.loc[hv]),
            "vn_mv_kv": float(net_start.bus.vn_kv.loc[mv]),
            "vn_lv_kv": float(net_start.bus.vn_kv.loc[lv]),
            "i0_hv_ka": float(i0_hv_ka_start) if np.isfinite(i0_hv_ka_start) else np.nan,
            "i0_mv_ka": float(i0_mv_ka_start) if np.isfinite(i0_mv_ka_start) else np.nan,
            "i0_lv_ka": float(i0_lv_ka_start) if np.isfinite(i0_lv_ka_start) else np.nan,
            "i0_max_ka": max(i0_hv_ka_start, i0_mv_ka_start, i0_lv_ka_start),
            "dI_hv_ka": float(dI_hv_ka) if np.isfinite(dI_hv_ka) else np.nan,
            "dI_mv_ka": float(dI_mv_ka) if np.isfinite(dI_mv_ka) else np.nan,
            "dI_lv_ka": float(dI_lv_ka) if np.isfinite(dI_lv_ka) else np.nan,
            "dI_max_ka": max(dI_hv_ka, dI_mv_ka, dI_lv_ka),
            "sf_hv": float(abs(sf_hv)) if np.isfinite(sf_hv) else np.nan,
            "sf_mv": float(abs(sf_mv)) if np.isfinite(sf_mv) else np.nan,
            "sf_lv": float(abs(sf_lv)) if np.isfinite(sf_lv) else np.nan,
            "sf_max": float(abs(np.nanmax([sf_hv, sf_mv, sf_lv]))),
        })

    return pd.DataFrame(rows).set_index("trafo3w_index")


def calc_impedance_current_sensitivity_from_power_flow(net_start, net_post, min_i_ka=1e-6):
    """
    Function to compute the sensitivity of impedance elements to a power change at the bus of interest. 
    Only impedances connecting buses at different voltage levels are taken into account.
    """

    rows = []
    for iidx, imp in net_start.impedance[net_start.impedance.in_service].iterrows():
        iidx = int(iidx)
        fb = int(imp.from_bus)
        tb = int(imp.to_bus)

        fv = net_start.bus.vn_kv.loc[fb]
        tv = net_start.bus.vn_kv.loc[tb]

        if fv == tv:
            continue

        # initial currents from PF
        i0_from_ka_start = abs(float(net_start.res_impedance.i_from_ka.loc[iidx])) if "i_from_ka" in net_start.res_impedance.columns else np.nan
        i0_to_ka_start = abs(float(net_start.res_impedance.i_to_ka.loc[iidx])) if "i_to_ka" in net_start.res_impedance.columns else np.nan

        # currents after perturbation from PF
        i0_from_ka_post = abs(float(net_post.res_impedance.i_from_ka.loc[iidx])) if "i_from_ka" in net_post.res_impedance.columns else np.nan
        i0_to_ka_post = abs(float(net_post.res_impedance.i_to_ka.loc[iidx])) if "i_to_ka" in net_post.res_impedance.columns else np.nan

        # current difference between before and after perturbation
        dI_from_ka = i0_from_ka_start - i0_from_ka_post
        dI_to_ka = i0_to_ka_start - i0_to_ka_post

        # sensitivity computation
        sf_from = dI_from_ka / max(i0_from_ka_start, float(min_i_ka)) if np.isfinite(dI_from_ka) else np.nan
        sf_to = dI_to_ka / max(i0_to_ka_start, float(min_i_ka)) if np.isfinite(dI_to_ka) else np.nan

        rows.append({
            "impedance_index": iidx,
            "from_bus": fb,
            "to_bus": tb,
            "vn_from_kv": float(net_start.bus.vn_kv.loc[fb]),
            "vn_to_kv": float(net_start.bus.vn_kv.loc[tb]),
            "i0_from_ka": float(i0_from_ka_start) if np.isfinite(i0_from_ka_start) else np.nan,
            "i0_to_ka": float(i0_to_ka_start) if np.isfinite(i0_to_ka_start) else np.nan,
            "i0_max_ka": max(i0_from_ka_start, i0_to_ka_start),
            "dI_from_ka": float(dI_from_ka) if np.isfinite(dI_from_ka) else np.nan,
            "dI_to_ka": float(dI_to_ka) if np.isfinite(dI_to_ka) else np.nan,
            "dI_max_ka": max(dI_from_ka, dI_to_ka),
            "sf_from": float(abs(sf_from)) if np.isfinite(sf_from) else np.nan,
            "sf_to": float(abs(sf_to)) if np.isfinite(sf_to) else np.nan,
            "sf_max": float(abs(np.nanmax([sf_from, sf_to]))),
        })

    if len(rows):
        return pd.DataFrame(rows).set_index("impedance_index")


def create_sets(net):
    """
    Function to identify the set of trafo, 3w-trafo and impedance to be considered in the sensitivity analysis.
    """

    el_pairs = set()
    el_adj = {}

    # ------------------------------------------------------------------
    # 2W trafos
    # ------------------------------------------------------------------
    for tidx, tr in net.trafo[net.trafo.in_service].iterrows():
        hv = int(tr.hv_bus)
        lv = int(tr.lv_bus)

        el_pairs.add(frozenset((hv, lv)))
        el_adj.setdefault(hv, []).append((lv, ("trafo", int(tidx))))
        el_adj.setdefault(lv, []).append((hv, ("trafo", int(tidx))))

    # ------------------------------------------------------------------
    # 3W trafos
    # ------------------------------------------------------------------
    for tidx, tr in net.trafo3w[net.trafo3w.in_service].iterrows():
        hv = int(tr.hv_bus)
        mv = int(tr.mv_bus)
        lv = int(tr.lv_bus)

        # all winding pairs exist electrically
        el_pairs.add(frozenset((hv, mv)))
        el_pairs.add(frozenset((hv, lv)))
        el_pairs.add(frozenset((mv, lv)))

        el_adj.setdefault(hv, []).append((mv, ("trafo3w", int(tidx))))
        el_adj.setdefault(hv, []).append((lv, ("trafo3w", int(tidx))))
        el_adj.setdefault(mv, []).append((hv, ("trafo3w", int(tidx))))
        el_adj.setdefault(mv, []).append((lv, ("trafo3w", int(tidx))))
        el_adj.setdefault(lv, []).append((hv, ("trafo3w", int(tidx))))
        el_adj.setdefault(lv, []).append((mv, ("trafo3w", int(tidx))))

    # ------------------------------------------------------------------
    # Impedances
    # ------------------------------------------------------------------
    for iidx, imp in net.impedance[net.impedance.in_service].iterrows():
        fb = int(imp.from_bus)
        tb = int(imp.to_bus)

        fv = net.bus.vn_kv.loc[fb]
        tv = net.bus.vn_kv.loc[tb]

        if fv == tv:
            continue
        
        if fv > tv:
            el_pairs.add(frozenset((fb, tb)))
        else:
            el_pairs.add(frozenset((tb, fb)))
        el_adj.setdefault(fb, []).append((tb, ("impedance", int(iidx))))
        el_adj.setdefault(tb, []).append((fb, ("impedance", int(iidx))))

    return el_pairs, el_adj


def cut_by_sensitivity(net, G, start_bus, trafo_sens_df, trafo3w_sens_df=None, impedance_sens_df=None,
                    sensitivity_threshold=0.05, min_working_current_ka=1e-4, dI_min_ka=1e-4, vn_max_kv=None, 
                    respect_switches=True, cut_downward_elements=True, keep_boundary_outside_bus=True):
    """
    Traversal from start_bus:
      - elements are cut if:
            sf_max < sensitivity_threshold
            OR dI_max_ka <= dI_min_ka
            OR to_vn > vn_max_kv

    Returns
    -------
    kept_buses : set[int]
    boundaries : list[dict]
    """
    start_bus = int(start_bus)
    vn = net.bus.vn_kv.astype(float)

    el_pairs, el_adj = create_sets(net)

    kept_buses = {start_bus}
    visited = {start_bus}
    queue = [start_bus]
    upper_boundaries = []
    lower_boundaries = []
    visited_element_dir = set()

    while queue:
        u = queue.pop(0)

        # --------------------------------------------------------------
        # 1) non-trafo (or impedance) neighbors
        # --------------------------------------------------------------
        for v in G.neighbors(u):
            v = int(v)
            if (frozenset((u, v)) in el_pairs) or (frozenset((v, u)) in el_pairs):
                continue
            if v not in visited:
                visited.add(v)
                kept_buses.add(v)
                queue.append(v)

        # --------------------------------------------------------------
        # 2) trafo or impedance neighbors
        # --------------------------------------------------------------
        for v, el_id in el_adj.get(u, []):
            v = int(v)
            if v > u:
                key = (el_id, int(u), int(v))
            else:
                key = (el_id, int(v), int(u))
            if key in visited_element_dir:
                continue
            visited_element_dir.add(key)

            vn_u = float(vn.loc[u])
            vn_v = float(vn.loc[v])

            # ==========================================================
            # 2W TRAFO
            # ==========================================================
            if el_id[0] == "trafo":
                tidx = int(el_id[1])

                # local upward traversal?
                is_upward = vn_v > vn_u + 1e-9

                if is_upward or cut_downward_elements:

                    # sensitivity data
                    if tidx in trafo_sens_df.index:
                        sf = float(trafo_sens_df.at[tidx, "sf_max"]) if "sf_max" in trafo_sens_df.columns else np.nan
                        i0 = float(trafo_sens_df.at[tidx, "i0_max_ka"]) if "i0_max_ka" in trafo_sens_df.columns else np.nan
                        dI = float(trafo_sens_df.at[tidx, "dI_max_ka"]) if "dI_max_ka" in trafo_sens_df.columns else np.nan
                    else:
                        sf = np.nan
                        i0 = np.nan
                        dI = np.nan

                    cut_due_to_vn = (
                        vn_max_kv is not None 
                        and vn_v > float(vn_max_kv) + 1e-9
                    )

                    cut_due_to_sens = (
                        np.isfinite(sf)
                        and np.isfinite(i0)
                        and i0 >= float(min_working_current_ka)
                        and sf < float(sensitivity_threshold)
                    )

                    cut_due_to_dI_min = (
                        np.isfinite(dI)
                        and abs(dI) < float(dI_min_ka)
                    )

                    if cut_due_to_vn or cut_due_to_sens or cut_due_to_dI_min:
                        tr = net.trafo.loc[tidx]
                        trafo_info = {
                            "el_type": "trafo",
                            "el_index": tidx,
                            "hv_bus": int(tr.hv_bus),
                            "lv_bus": int(tr.lv_bus),
                            "boundary_bus_inside": int(u),
                            "boundary_bus_outside": int(v),
                            "reason": "vn_above_vn_max" if cut_due_to_vn else "up_below_current_sensitivity",
                            "from_vn_kv": vn_u,
                            "to_vn_kv": vn_v,
                            "sf_max": sf,
                            "i0_max_ka": i0,
                            "dI_max_ka": dI,
                            "sensitivity_threshold": float(sensitivity_threshold),
                            "min_working_current_ka": float(min_working_current_ka),
                            "dI_min_ka": float(dI_min_ka),
                        }
                        if is_upward:
                            upper_boundaries.append(trafo_info)
                        else:
                            lower_boundaries.append(trafo_info)

                        if keep_boundary_outside_bus:
                            kept_buses.add(int(v))
                        continue

                if v not in visited:
                    visited.add(v)
                    kept_buses.add(v)
                    queue.append(v)

                continue

            # ==========================================================
            # 3W TRAFO
            # ==========================================================
            if el_id[0] == "trafo3w":
                tidx = int(el_id[1])
                tr3 = net.trafo3w.loc[tidx]

                hv = int(tr3.hv_bus)
                mv = int(tr3.mv_bus)
                lv = int(tr3.lv_bus)

                if hv not in {u, v}: 
                    z = hv
                elif mv not in {u, v}:
                    z = mv
                else:
                    z = lv
                vn_z = float(vn.loc[z])

                if z > u:
                    key = (el_id, int(u), int(z))
                else:
                    key = (el_id, int(z), int(u))
                visited_element_dir.add(key)

                if v > z:
                    key = (el_id, int(z), int(v))
                else:
                    key = (el_id, int(v), int(z))
                visited_element_dir.add(key)

                is_upward = (vn_v > vn_u + 1e-9) or (vn_z > vn_u + 1e-9)

                if is_upward or cut_downward_elements:

                    if trafo3w_sens_df is not None and tidx in trafo3w_sens_df.index:
                        sf = float(trafo3w_sens_df.at[tidx, "sf_max"]) if "sf_max" in trafo3w_sens_df.columns else np.nan
                        i0 = float(trafo3w_sens_df.at[tidx, "i0_max_ka"]) if "i0_max_ka" in trafo3w_sens_df.columns else np.nan
                        dI = float(trafo3w_sens_df.at[tidx, "dI_max_ka"]) if "dI_max_ka" in trafo3w_sens_df.columns else np.nan
                    else:
                        sf = np.nan
                        i0 = np.nan
                        dI = np.nan

                    cut_due_to_vn = (
                        vn_max_kv is not None
                        and float(vn.loc[hv]) > float(vn_max_kv) + 1e-9
                    )

                    cut_due_to_sens = (
                        np.isfinite(sf)
                        and np.isfinite(i0)
                        and i0 >= float(min_working_current_ka)
                        and sf < float(sensitivity_threshold)
                    )

                    cut_due_to_dI_min = (
                        np.isfinite(dI)
                        and abs(dI) < float(dI_min_ka)
                    )

                    if cut_due_to_vn or cut_due_to_sens or cut_due_to_dI_min:
                        trafo_info = {
                            "el_type": "trafo3w",
                            "el_index": tidx,
                            "hv_bus": hv,
                            "mv_bus": mv,
                            "lv_bus": lv,
                            "boundary_bus_inside": int(u),  
                            "boundary_bus_outside": int(v),
                            "boundary_bus_other": int(z),
                            "reason": "vn_above_vn_max" if cut_due_to_vn else "up_below_current_sensitivity",
                            "from_vn_kv": vn_u,
                            "to_vn_kv": vn_v,
                            "sf_max": sf,
                            "i0_max_ka": i0,
                            "dI_max_ka": dI,
                            "sensitivity_threshold": float(sensitivity_threshold),
                            "min_working_current_ka": float(min_working_current_ka),
                            "dI_min_ka": float(dI_min_ka),
                        }
                        if is_upward:
                            upper_boundaries.append(trafo_info)
                        else:
                            lower_boundaries.append(trafo_info)

                        if keep_boundary_outside_bus:
                            kept_buses.add(int(v))
                            kept_buses.add(int(z))
                        continue

                if v not in visited:
                    visited.add(v)
                    visited.add(z)
                    kept_buses.add(v)
                    kept_buses.add(z)
                    queue.append(v)
                    queue.append(z)

                continue

            # ==========================================================
            # IMPEDANCE
            # ==========================================================
            if el_id[0] == "impedance":
                iidx = int(el_id[1])

                # local upward traversal?
                is_upward = vn_v > vn_u + 1e-9

                if is_upward or cut_downward_elements:

                    # sensitivity data
                    if iidx in impedance_sens_df.index:
                        sf = float(impedance_sens_df.at[iidx, "sf_max"]) if "sf_max" in impedance_sens_df.columns else np.nan
                        i0 = float(impedance_sens_df.at[iidx, "i0_max_ka"]) if "i0_max_ka" in impedance_sens_df.columns else np.nan
                        dI = float(impedance_sens_df.at[iidx, "dI_max_ka"]) if "dI_max_ka" in impedance_sens_df.columns else np.nan
                    else:
                        sf = np.nan
                        i0 = np.nan
                        dI = np.nan

                    cut_due_to_vn = (
                        vn_max_kv is not None
                        and vn_v > float(vn_max_kv) + 1e-9
                    )

                    cut_due_to_sens = (
                        np.isfinite(sf)
                        and np.isfinite(i0)
                        and i0 >= float(min_working_current_ka)
                        and sf < float(sensitivity_threshold)
                    )

                    cut_due_to_dI_min = (
                        np.isfinite(dI)
                        and abs(dI) < float(dI_min_ka)
                    )

                    if cut_due_to_vn or cut_due_to_sens or cut_due_to_dI_min:
                        if is_upward:
                            hv_bus = v
                            lv_bus = u
                        else:
                            hv_bus = u
                            lv_bus = v

                        imp_info = {
                            "el_type": "impedance",
                            "el_index": iidx,
                            "hv_bus": hv_bus,
                            "lv_bus": lv_bus,
                            "boundary_bus_inside": int(u),
                            "boundary_bus_outside": int(v),
                            "reason": "vn_above_vn_max" if cut_due_to_vn else "up_below_current_sensitivity",
                            "from_vn_kv": vn_u,
                            "to_vn_kv": vn_v,
                            "sf_max": sf,
                            "i0_max_ka": i0,
                            "dI_max_ka": dI,
                            "sensitivity_threshold": float(sensitivity_threshold),
                            "min_working_current_ka": float(min_working_current_ka),
                            "dI_min_ka": float(dI_min_ka),
                        }
                        if is_upward:
                            upper_boundaries.append(imp_info)
                        else:
                            lower_boundaries.append(imp_info)

                        continue

                if v not in visited:
                    visited.add(v)
                    kept_buses.add(v)
                    queue.append(v)

                continue

    return kept_buses, upper_boundaries, lower_boundaries


def find_trafo_from_ext_grid(net, subnet, G):
    """
    This function is used if no external grid exists in the reduced grid.
    It finds the transformer connected to the external grid in the original model.
    """

    el_pairs, el_adj = create_sets(net)
    
    queue = net.ext_grid["bus"].tolist()
    visited = set(queue)
    visited_element_dir = set()
    created = set()

    while queue:
        u = queue.pop(0)

        # --------------------------------------------------------------
        # 1) non-trafo neighbors
        # --------------------------------------------------------------
        for v in G.neighbors(u):
            v = int(v)
            if frozenset((u, v)) in el_pairs:
                continue
            if v not in visited:
                visited.add(v)
                queue.append(v)

        # --------------------------------------------------------------
        # 2) neighbors
        # --------------------------------------------------------------
        for v, el_id in el_adj.get(u, []):
            v = int(v)
            if v > u:
                key = (el_id, int(u), int(v))
            else:
                key = (el_id, int(v), int(u))
            if key in visited_element_dir:
                continue
            visited_element_dir.add(key)

            # ==========================================================
            # 2W TRAFO
            # ==========================================================
            if el_id[0] == "trafo":
                tidx = int(el_id[1])

                subnet_boundary_trafo = subnet.trafo[subnet.trafo.index == tidx]
                if subnet_boundary_trafo.empty:
                    if v not in visited:
                        visited.add(v)
                        queue.append(v)
                else:
                    vm = float(net.res_bus.vm_pu.loc[u])
                    va = float(net.res_bus.va_degree.loc[u])
                    pp.create_ext_grid(subnet, bus=u, vm_pu=vm, va_degree=va)
                    created.add(u)

            # ==========================================================
            # 3W TRAFO
            # ==========================================================
            if el_id[0] == "trafo3w":
                tidx = int(el_id[1])

                subnet_boundary_trafo3w = subnet.trafo3w[subnet.trafo3w.index == tidx]
                if subnet_boundary_trafo3w.empty:
                    if v not in visited:
                        visited.add(v)
                        queue.append(v)
                else:
                    vm = float(net.res_bus.vm_pu.loc[u])
                    va = float(net.res_bus.va_degree.loc[u])
                    pp.create_ext_grid(subnet, bus=u, vm_pu=vm, va_degree=va)
                    created.add(u)

            # ==========================================================
            # IMPEDANCE
            # ==========================================================
            if el_id[0] == "impedance":
                iidx = int(el_id[1])

                subnet_boundary_impedance = subnet.impedance[subnet.impedance.index == iidx]
                if subnet_boundary_impedance.empty:
                    if v not in visited:
                        visited.add(v)
                        queue.append(v)
                else:
                    vm = float(net.res_bus.vm_pu.loc[u])
                    va = float(net.res_bus.va_degree.loc[u])
                    pp.create_ext_grid(subnet, bus=u, vm_pu=vm, va_degree=va)
                    created.add(u)

    return subnet, created


def add_boundary_ext_grids(subnet, net, boundaries, subG, G):
    """
    Creates ext_grids at the boundary buses that remains inside the subnet.
    It considers the boundary buses met in upstream direction.
    """

    created = set()
    created_pq = set()
    boundary_list = set()

    for bnd in boundaries:
        
        btype = bnd["el_type"]

        if (btype == "trafo3w") or (btype == "trafo"):
            b = int(bnd["hv_bus"])
        else: 
            b = int(bnd["lv_bus"])

        v = [v for v in subG.neighbors(b)]
        boundary_list.add(b)

        vm = float(net.res_bus.vm_pu.loc[b])
        va = float(net.res_bus.va_degree.loc[b])

        if b not in created:
            pp.create_ext_grid(subnet, bus=b, vm_pu=vm, va_degree=va)
            created.add(b)

        if btype == "trafo3w":
            b_pq = bnd["boundary_bus_other"]         
            v = [v for v in subG.neighbors(b_pq)]  

            if len(v) == 2: 
                idx = bnd["el_index"]
                if bnd["mv_bus"] == b_pq:
                    p = net.res_trafo3w.p_mv_mw.loc[idx]
                    q = net.res_trafo3w.q_mv_mvar.loc[idx]
                else:
                    p = net.res_trafo3w.p_lv_mw.loc[idx]
                    q = net.res_trafo3w.q_lv_mvar.loc[idx]

                pp.create_sgen(subnet, bus=b_pq, p_mw=p, q_mvar=q)
                created_pq.add(b_pq)

    if subnet.ext_grid.empty:
        subnet, b = find_trafo_from_ext_grid(net, subnet, G)
        created = created.union(b)

    return created, created_pq


def compensate_pq_inj_for_elements_connected_to_bus(net, pq, b, str):
    """
    This function compensates for already existing loads, sgens, or other power injection elements
    already existing at the boundary bus.
    """

    def compensate_pq(net, b, str, element):
        res_el = "res_" + element

        if np.any(net[element][net[element].bus==b]):
            if str == "active":
                val = net[res_el].p_mw[net[element].bus==b].sum()
            elif str == "reactive":
                val = net[res_el].q_mvar[net[element].bus==b].sum()
        else:
            val = 0

        return val

    pq += compensate_pq(net, b, str, "load")        # compensation for connected loads
    pq -= compensate_pq(net, b, str, "sgen")        # compensation for connected sgens
    pq -= compensate_pq(net, b, str, "gen")         # compensation for connected gens
    pq += compensate_pq(net, b, str, "shunt")       # compensation for connected shunts
    pq += compensate_pq(net, b, str, "ward")        # compensation for connected ward
    pq += compensate_pq(net, b, str, "xward")       # compensation for connected xward

    return pq


def add_boundary_pq_injections(subnet, net, boundaries, subG):
    """
    Creates PQ injections at the boundary bus that remains inside the subnet.
    It considers the boundary buses met in downstream direction.
    """

    created = set()

    for bnd in boundaries:

        btype = bnd["el_type"]
        if btype == "trafo":
            b = int(bnd["lv_bus"])
            tr_idx = int(bnd["el_index"])

            v = [v for v in subG.neighbors(b)]
            create_PQ = True
            if len(v)>1:
                b_volt = subnet.bus.vn_kv.loc[b]
                for it in v:
                    v_volt = subnet.bus.vn_kv.loc[it]
                    if v_volt == b_volt:
                        create_PQ = False

            if create_PQ:
                p = float(net.res_trafo.p_lv_mw.loc[tr_idx])
                q = float(net.res_trafo.q_lv_mvar.loc[tr_idx])

                if b not in created:
                    p = compensate_pq_inj_for_elements_connected_to_bus(net, p, b, "active")
                    q = compensate_pq_inj_for_elements_connected_to_bus(net, q, b, "reactive")

                pp.create_sgen(subnet, bus=b, p_mw=p, q_mvar=q)
                created.add(b)
            
        elif bnd["el_type"] == "trafo3w":
            b_mv = int(bnd["mv_bus"])
            b_lv = int(bnd["lv_bus"])
            tr_idx = int(bnd["el_index"])

            v_mv = [v for v in subG.neighbors(b_mv)]
            if len(v_mv)>2:
                continue

            v_lv = [v for v in subG.neighbors(b_lv)]
            if len(v_lv)>2:
                continue

            p_mv = float(net.res_trafo3w.p_mv_mw.loc[tr_idx])
            q_mv = float(net.res_trafo3w.q_mv_mvar.loc[tr_idx])
            p_lv = float(net.res_trafo3w.p_lv_mw.loc[tr_idx])
            q_lv = float(net.res_trafo3w.q_lv_mvar.loc[tr_idx])

            if b_mv not in created:
                p_mv = compensate_pq_inj_for_elements_connected_to_bus(net, p_mv, b_mv, "active")
                q_mv = compensate_pq_inj_for_elements_connected_to_bus(net, q_mv, b_mv, "reactive")

            if b_lv not in created:
                p_lv = compensate_pq_inj_for_elements_connected_to_bus(net, p_lv, b_lv, "active")
                q_lv = compensate_pq_inj_for_elements_connected_to_bus(net, q_lv, b_lv, "reactive")
            
            pp.create_sgen(subnet, bus=b_mv, p_mw=p_mv, q_mvar=q_mv)
            pp.create_sgen(subnet, bus=b_lv, p_mw=p_lv, q_mvar=q_lv)
            created.add(b_mv)
            created.add(b_lv)

        else:
            b = int(bnd["hv_bus"])
            imp_idx = int(bnd["el_index"])

            b_lv = int(bnd["lv_bus"])
            if np.any(subnet.bus.index==b_lv):
                continue
            
            if net.impedance.from_bus.loc[imp_idx] == b:
                p = - float(net.res_impedance.p_from_mw.loc[imp_idx])
                q = - float(net.res_impedance.q_from_mvar.loc[imp_idx])
            else:
                p = - float(net.res_impedance.p_to_mw.loc[imp_idx])
                q = - float(net.res_impedance.q_to_mvar.loc[imp_idx])

            pp.create_sgen(subnet, bus=b, p_mw=p, q_mvar=q)
            created.add(b)

    return created


def build_reduced_network(net, start_bus, method="power flow", sensitivity_threshold=0.05, 
                        min_working_current_ka=0.01, vn_max_kv=None, deltaP_MW=1.0, deltaQ_Mvar=0.0, 
                        cut_downward_elements=True):
    """
    Complete workflow based on classical bus sensitivity calculation:
      1) assumes base PF already exists in net
      2) runs power flows with pwr inj variation
      3) computes element sensitivities
      4) cuts elements by sensitivity / vn_max / i_min
      5) builds subnet
      6) adds boundary ext_grids or power injections

    Returns
    -------
    subnet, trafo_sens_df, trafo3w_sens_df, impedance_sens_df, kept_buses, boundaries, created_ext_grids, created_pq_injections
    """

    if ~np.any(net.bus.index == start_bus):
        print("The selected bus was not found in the considered grid")
        return net, np.empty, np.empty(0), np.empty(0), np.empty(0), np.empty(0), np.empty(0)

    net_post = copy.deepcopy(net)
    pp.runpp(net, run_control=False, max_iteration=100)

    pp.create_sgen(net_post, bus=start_bus, p_mw=deltaP_MW, q_mvar=deltaQ_Mvar)
    pp.runpp(net_post, run_control=False, max_iteration=100)

    trafo_sens_df = calc_trafo_current_sensitivity_from_power_flow(
        net,
        net_post,
        min_i_ka=1e-6)
    
    trafo3w_sens_df = calc_trafo3w_current_sensitivity_from_power_flow(
        net,
        net_post,
        min_i_ka=1e-6)
    
    impedance_sens_df = calc_impedance_current_sensitivity_from_power_flow(
        net,
        net_post,
        min_i_ka=1e-6)
    
    G = top.create_nxgraph(net, respect_switches=True)

    kept_buses, hv_boundaries, lv_boundaries = cut_by_sensitivity(
        net, G,
        start_bus=start_bus,
        trafo_sens_df=trafo_sens_df,
        trafo3w_sens_df=trafo3w_sens_df,
        impedance_sens_df=impedance_sens_df,
        sensitivity_threshold=sensitivity_threshold,
        min_working_current_ka=min_working_current_ka,
        dI_min_ka=1e-4,
        vn_max_kv=vn_max_kv,
        respect_switches=True,
        cut_downward_elements=cut_downward_elements,
        keep_boundary_outside_bus=True)

    subnet = select_subnet(net, buses=list(kept_buses), include_results=True)
    subnet.user_pf_options = net.user_pf_options
    subG = top.create_nxgraph(subnet, respect_switches=True)

    created_ext_grids, created_pq = add_boundary_ext_grids(subnet, net, hv_boundaries, subG, G)
    created_pq_injections = add_boundary_pq_injections(subnet, net, lv_boundaries, subG)
    created_pq_injections = created_pq_injections.union(created_pq)

    boundaries = {}
    boundaries["hv"] = hv_boundaries
    boundaries["lv"] = lv_boundaries

    try:
        trafo_sens_sorted = trafo_sens_df.sort_values(by="sf_max", ascending=False)
    except:
        trafo_sens_sorted = None
    try:
        trafo3w_sens_sorted = trafo3w_sens_df.sort_values(by="sf_max", ascending=False)
    except: 
        trafo3w_sens_sorted = None
    try:
        impedance_sens_sorted = impedance_sens_df.sort_values(by="sf_max", ascending=False)
    except:
        impedance_sens_sorted = None

    return subnet, trafo_sens_sorted, trafo3w_sens_sorted, impedance_sens_sorted, boundaries, created_ext_grids, created_pq_injections

#### Measurement creation
The following blocks of code implement the functions necessary to create some measurements in the grid, which are needed for the state estimation or state forecasting purposes. 

To emulate real-time state estimation, only a limited number of measurements will be created. 
These will be voltage measurements and active and reactive power measurements at the external grid buses. 
For forecasting purposes, power injection forecast data could be instead available to grid operators. 

The following code will implement the funcions to create such measurements as well as the function to add uncertainty to these measurements, which emulates the limited accuracy of measurement instruments (or of forecast data) in the grid. 

In [ ]:
def add_measurement_uncertainty(meas, unc):
    dev = abs(unc/300*meas)
    for m in range(len(meas)):
        meas[m] = np.random.normal(meas[m], dev[m])
    return meas


def create_ext_grid_voltage_measurements(net, unc):
    idx = net.ext_grid.bus
    v_values = net.ext_grid["vm_pu"].values
    v_values = add_measurement_uncertainty(v_values, unc)

    measv = ["v"]*len(v_values)
    element = ["bus"]*len(v_values)
    std_dev_v = [unc/300]*len(v_values)

    measV = pd.DataFrame({"measurement_type":measv, 
                        "element_type":element,
                        "element":idx,
                        "value":v_values.tolist(),
                        "std_dev":std_dev_v})
    
    net["measurement"] = pd.concat([net["measurement"],measV], ignore_index=True)
    return net


def create_ext_grid_inj_measurements(net, unc):
    idx = net.ext_grid.bus
    p_values = net.res_bus["p_mw"].loc[net.ext_grid.bus].values
    q_values = net.res_bus["q_mvar"].loc[net.ext_grid.bus].values
    p_values = add_measurement_uncertainty(p_values, unc)
    q_values = add_measurement_uncertainty(q_values, unc)

    measp = ["p"]*len(p_values)
    measq = ["q"]*len(q_values)
    element = ["bus"]*len(q_values)
    std_dev_p = [unc/300]*len(p_values)
    std_dev_q = [unc/300]*len(q_values)

    measP = pd.DataFrame({"measurement_type":measp, 
                        "element_type":element,
                        "element":idx,
                        "value":p_values.tolist(),
                        "std_dev":std_dev_p})
    measQ = pd.DataFrame({"measurement_type":measq, 
                        "element_type":element,
                        "element":idx,
                        "value":q_values.tolist(),
                        "std_dev":std_dev_q})
       
    net["measurement"] = pd.concat([net["measurement"],measP,measQ], ignore_index=True)
    return net


def create_bus_inj_measurements(net, unc):
    idx = net.bus.index
    p_values = net.res_bus["p_mw"].values
    q_values = net.res_bus["q_mvar"].values
    p_values = add_measurement_uncertainty(p_values, unc)
    q_values = add_measurement_uncertainty(q_values, unc)

    measp = ["p"]*len(p_values)
    measq = ["q"]*len(q_values)
    element = ["bus"]*len(q_values)
    std_dev_p = [unc/300]*len(p_values)
    std_dev_q = [unc/300]*len(q_values)

    measP = pd.DataFrame({"measurement_type":measp, 
                        "element_type":element,
                        "element":idx,
                        "value":p_values.tolist(),
                        "std_dev":std_dev_p})
    measQ = pd.DataFrame({"measurement_type":measq, 
                        "element_type":element,
                        "element":idx,
                        "value":q_values.tolist(),
                        "std_dev":std_dev_q})

    net["measurement"] = pd.concat([net["measurement"],measP,measQ], ignore_index=True)
    return net

#### UK Power Network grids
This tutorial assumes that the grids of UK Power Networks have been already imported from the CIM data and saved as pandapower networks in json format. 
To see how to import the UK Power Networks grids starting from the CIM files downloadable from the UK Power Networks portal, please refer to the following [UKPN_CIM2pp_tutorial](). 
Here you can also find how to save the pandapower grid into a json file and how to navigate through the pandapower grid data or the attributes of the different grid components. 

### State Estimation, Example 1 - LPN grid with very low measurement uncertainty

In this first example, state estimation will be shown on an exemplary portion of the LPN grid. 

A very low measurement uncertainty will be applied to test the grid with ideal conditions (measurements very close to true values)

In [ ]:
# Import the grid for the analysis
filename = "LPN EQ SSH_0401_eq.json"   # Give here the name of the json file with the UKPN grid you want to use
if os.path.isfile(filename):
    net = pp.from_json(filename)
else:
    print("file does not exist, creating a dummy net")
    net = pp.create_empty_network()
    bus = pp.create_bus(net, vn_kv=132)
    pp.create_ext_grid(net, bus=bus)

In [ ]:
# Apply the workarounds on the selected grid
license_area = "LPN"  # Provide here the name of the considered license area. It should be "LPN", "SPN", or "EPN".
if net.bus.index.size > 1:
    remove_impedance = True     # Decide if removing fictious impedances from the grid or not
    net = apply_workarounds(net, license_area, remove_impedance)

Apply the grid reduction to focus the analysis on a limited portion of the overall network

In [ ]:
start_bus = 1712    # Select the bus of interest around which you want to reduce the grid

# Call the main function for grid reduction
subnet, trafo_sens_df, trafo3w_sens_df, impedance_sens_df, boundaries, created_ext_grids, created_pq_injections = build_reduced_network(
    net, 
    start_bus=start_bus,            # start bus considered for the reduction
    method="power_flow",            # used method (only power flow available in this tutorial)
    sensitivity_threshold=0.05,     # threshold to decide if cutting or not the subnet
    min_working_current_ka=0.001,   # minimum current limit considered for the cutting
    vn_max_kv=50.0,                 # maximum voltage limit considered for the cutting
    deltaP_MW=1.0,                  # delta of active power toapplied for the sensitivity calculation
    deltaQ_Mvar=0.0,                # delta of reactive power toapplied for the sensitivity calculation
    cut_downward_elements=True)     # decide if apply cuts also in downstream direction (lower voltage levels) or not

if ~np.any(net.bus.index == start_bus):
    print("The selected bus was not found in the considered grid")
else:
    print("kept buses:", len(subnet.bus))
    print("boundaries:", len(boundaries["hv"])+len(boundaries["lv"]))
    print("ext_grids created:", len(created_ext_grids))
    print("pq_injections_created:", len(created_pq_injections))

Run a power flow to create the conditions of the grid assumed as reference values and extract the measurements from such conditions.

In [ ]:
# Run the power flow
pp.runpp(subnet, run_control=False, lightsim2grid=False, max_iteration=100)

# Clean measurements already present in the grid
subnet.measurement.drop(subnet.measurement.index, inplace=True)

# Create the voltage measurements with 0.01% uncertainty
subnet = create_ext_grid_voltage_measurements(subnet, unc=0.01)

# Create the active and reactive power measurements with 0.01% uncertainty
subnet = create_ext_grid_inj_measurements(subnet, unc=0.01)

# Remove injection values from shunt or ward elements (which would not be seen as power injection measurements)
remove_shunt_injection_from_meas(subnet,"shunt")
remove_shunt_injection_from_meas(subnet,"ward")

Visualize the measurements that will be used for state estimation

In [ ]:
display(subnet.measurement)
display("Total nodes in the grid: " + str(len(subnet.bus)))
display("Total number of measurements: " + str(len(subnet.measurement)))
display("Measurement redundancy: " + "{:.2f}".format(100*len(subnet.measurement)/(2*len(subnet.bus))) + " %")

Create load and generation clusters needed for the AF-WLS algorithm.

**Note**: here, for the sake of simplicity, only one cluster for the loads and one cluster for the generators is created. In general, however, the algorithm is able to work with multiple clusters (e.g., residential, commercial, industrial, etc. for loads, or PV, wind, etc. for generation). Those clusters are often available in real grids and are necessary for using the AF-WLS algorithm. In pandapower, those clusters should be assigned within the "type" attribute of loads and sgens.

In [ ]:
# Create the load and generation clusters under the load and sgen "type" attribute
subnet.load["type"] = "generic_load"
subnet.sgen["type"] = "generic_sgen"

Modify the nominal values of P and Q for loads and sgens.

**Note**: differently from power flow calculations, state estimation relies only on the use of measurements (provided in the net.measurement table). In the AF-WLS algorithm, the powers of loads and sgens are considered as nominal values, which are adopted to derive the allocation factors estimated in the algorithm. Modifying the values of P and Q, as done in the following block of code, allows therefore to change the "nominal" values of power of loads and sgens with respect to those previously considered for creating the reference conditions via the power flow. 

In [ ]:
# Modify the nominal values of power
subnet.load["p_mw"] *= 2
subnet.load["q_mvar"] *= 2
subnet.sgen["p_mw"] *= 4
subnet.sgen["q_mvar"] *= 4

Run state estimation

In [ ]:
# Run the state estimation algorithm
try: 
    success = se.estimate(subnet, algorithm="af-wls", tolerance=1e-4, maximum_iterations=500)
    display("State estimation successfully converged in " + str(success["num_iterations"]) + " iterations.")
except: 
    display("State estimation did not converge")

Compare voltage magnitude state estimation results to reference results given by the initial power flow:

In [ ]:
if subnet.res_bus_est.index.size > 1: 
    volt_diff = subnet.res_bus["vm_pu"].values - subnet.res_bus_est["vm_pu"].values
    max_volt_diff = np.max(abs(volt_diff))
    display("Maximum voltage difference = " + "{:.6f}".format(max_volt_diff) + " p.u.")

Compare line current magnitude state estimation results to reference results given by the initial power flow:

In [ ]:
if subnet.res_line_est.index.size > 1: 
    curr_diff = subnet.res_line["i_from_ka"].values - subnet.res_line_est["i_from_ka"].values
    max_curr_diff = np.max(abs(curr_diff))
    display("Maximum current difference = " + "{:.6f}".format(max_curr_diff) + " kA")

Visualize estimated allocation factors:

In [ ]:
if hasattr(subnet, "res_cluster_est"):
    display("Estimated load allocation factor = " + "{:.2f}".format(subnet.res_cluster_est[0]))
    display("Estimated generation allocation factor = " + "{:.2f}".format(subnet.res_cluster_est[1]))

### State Estimation, Example 2 - LPN grid with realistic measurement uncertainty

In this example, state estimation will be shown on the same portion of the LPN grid used in example 1, but measurement uncertainties will be modified to reflect more realistic conditions usually present in real grids.

In [ ]:
# Clean measurements already present in the grid
subnet.measurement.drop(subnet.measurement.index, inplace=True)

# Create the voltage measurements with 0.2% uncertainty
subnet = create_ext_grid_voltage_measurements(subnet, unc=0.2)

# Create the active and reactive power measurements with 1% uncertainty
subnet = create_ext_grid_inj_measurements(subnet, unc=1)

# Remove injection values from shunt or ward elements (which would not be seen as power injection measurements)
remove_shunt_injection_from_meas(subnet,"shunt")
remove_shunt_injection_from_meas(subnet,"ward")

Visualize the measurements used for state estimation

In [ ]:
display(subnet.measurement)

Run state estimation

In [ ]:
# Run the state estimation algorithm
try: 
    success = se.estimate(subnet, algorithm="af-wls", tolerance=1e-4, maximum_iterations=500)
    display("State estimation successfully converged in " + str(success["num_iterations"]) + " iterations.")
except: 
    display("State estimation did not converge")

Compare the voltage magnitude state estimation results to the reference results given by the initial power flow; in this case, higher errors should be expected with respect to example 1, due to the larger uncertainty of the measurements. 

In [ ]:
if subnet.res_bus_est.index.size > 1: 
    volt_diff = subnet.res_bus["vm_pu"].values - subnet.res_bus_est["vm_pu"].values
    max_volt_diff = np.max(abs(volt_diff))
    display("Maximum voltage difference = " + "{:.6f}".format(max_volt_diff) + " p.u.")

Compare line current magnitude state estimation results to reference results given by the initial power flow; in this case, higher errors should be expected with respect to example 1, due to the larger uncertainty of the measurements. 

In [ ]:
if subnet.res_line_est.index.size > 1: 
    curr_diff = subnet.res_line["i_from_ka"].values - subnet.res_line_est["i_from_ka"].values
    max_curr_diff = np.max(abs(curr_diff))
    display("Maximum current difference = " + "{:.6f}".format(max_curr_diff) + " kA")

Visualize the estimated allocation factors:

In [ ]:
if hasattr(subnet, "res_cluster_est"):
    display("Estimated load allocation factor = " + "{:.2f}".format(subnet.res_cluster_est[0]))
    display("Estimated generation allocation factor = " + "{:.2f}".format(subnet.res_cluster_est[1]))

### State Estimation, Example 3 - SPN grid (realistic measurement uncertainty)

In this example, state estimation will be shown on a portion of the SPN grid. Realistic measurement uncertainties will be considered. 

In [ ]:
# Import the grid for the analysis
filename = "SPN EQ SSH_0401_eq.json"   # Give here the name of the json file with the UKPN grid you want to use
if os.path.isfile(filename):
    net = pp.from_json(filename)
else:
    print("file does not exist, creating a dummy net")
    net = pp.create_empty_network()
    bus = pp.create_bus(net, vn_kv=132)
    pp.create_ext_grid(net, bus=bus)

In [ ]:
# Apply the workarounds on the selected grid
license_area = "SPN"  # Provide here the name of the considered license area. It should be "LPN", "SPN", or "EPN".
if net.bus.index.size > 1:
    remove_impedance = True     # Decide if removing fictious impedances from the grid or not
    net = apply_workarounds(net, license_area, remove_impedance)

Apply the grid reduction to focus the analysis on a limited portion of the overall network

In [ ]:
start_bus = 4101    # Select the bus of interest around which you want to reduce the grid

# Call the main function for grid reduction
subnet, trafo_sens_df, trafo3w_sens_df, impedance_sens_df, boundaries, created_ext_grids, created_pq_injections = build_reduced_network(
    net, 
    start_bus=start_bus,            # start bus considered for the reduction
    method="power_flow",            # used method (only power flow available in this tutorial)
    sensitivity_threshold=0.05,     # threshold to decide if cutting or not the subnet
    min_working_current_ka=0.001,   # minimum current limit considered for the cutting
    vn_max_kv=50.0,                 # maximum voltage limit considered for the cutting
    deltaP_MW=1.0,                  # delta of active power toapplied for the sensitivity calculation
    deltaQ_Mvar=0.0,                # delta of reactive power toapplied for the sensitivity calculation
    cut_downward_elements=True)     # decide if apply cuts also in downstream direction (lower voltage levels) or not

if ~np.any(net.bus.index == start_bus):
    print("The selected bus was not found in the considered grid")
else:
    print("kept buses:", len(subnet.bus))
    print("boundaries:", len(boundaries["hv"])+len(boundaries["lv"]))
    print("ext_grids created:", len(created_ext_grids))
    print("pq_injections_created:", len(created_pq_injections))

Run the power flow to create the conditions of the grid assumed as reference values and extract the measurements from such conditions.

In [ ]:
# Run the power flow
pp.runpp(subnet, run_control=False, lightsim2grid=False, max_iteration=100)

# Clean measurements already present in the grid
subnet.measurement.drop(subnet.measurement.index, inplace=True)

# Create the voltage measurements with 0.2% uncertainty
subnet = create_ext_grid_voltage_measurements(subnet, unc=0.2)

# Create the active and reactive power measurements 1% uncertainty
subnet = create_ext_grid_inj_measurements(subnet, unc=1)

# Remove injection values from shunt or ward elements (which would not be seen as power injection measurements)
remove_shunt_injection_from_meas(subnet,"shunt")
remove_shunt_injection_from_meas(subnet,"ward")

Visualize the measurements that will be used for state estimation purposes

In [ ]:
display(subnet.measurement)
display("Total nodes in the grid: " + str(len(subnet.bus)))
display("Total number of measurements: " + str(len(subnet.measurement)))
display("Measurement redundancy: " + "{:.2f}".format(100*len(subnet.measurement)/(2*len(subnet.bus))) + " %")

Create the load and generation clusters needed for the AF-WLS algorithm

In [ ]:
# Create the load and generation clusters under the load and sgen "type" attribute
subnet.load["type"] = "generic_load"
subnet.sgen["type"] = "generic_sgen"

Modify the nominal values of P and Q for loads and sgens

In [ ]:
# Modify the nominal values of power
subnet.load["p_mw"] *= 5
subnet.load["q_mvar"] *= 5
subnet.sgen["p_mw"] *= 3
subnet.sgen["q_mvar"] *= 3

Run state estimation

In [ ]:
# Run the state estimation algorithm
try: 
    success = se.estimate(subnet, algorithm="af-wls", tolerance=1e-4, maximum_iterations=500)
    display("State estimation successfully converged in " + str(success["num_iterations"]) + " iterations.")
except: 
    display("State estimation did not converge")

Compare voltage magnitude state estimation results to reference results given by the initial power flow:

In [ ]:
if subnet.res_bus_est.index.size > 1: 
    volt_diff = subnet.res_bus["vm_pu"].values - subnet.res_bus_est["vm_pu"].values
    max_volt_diff = np.max(abs(volt_diff))
    display("Maximum voltage difference = " + "{:.6f}".format(max_volt_diff) + " p.u.")

Compare line current magnitude state estimation results to reference results given by the initial power flow:

In [ ]:
if subnet.res_line_est.index.size > 1: 
    curr_diff = subnet.res_line["i_from_ka"].values - subnet.res_line_est["i_from_ka"].values
    max_curr_diff = np.max(abs(curr_diff))
    display("Maximum current difference = " + "{:.6f}".format(max_curr_diff) + " kA")

Visualize the estimated allocation factors:

In [ ]:
if hasattr(subnet, "res_cluster_est"):
    display("Estimated load allocation factor = " + "{:.2f}".format(subnet.res_cluster_est[0]))
    display("Estimated generation allocation factor = " + "{:.2f}".format(subnet.res_cluster_est[1]))

### State Estimation, Example 4 - EPN grid (realistic measurement uncertainty)

In this example, state estimation will be shown on a portion of the EPN grid. Realistic measurement uncertainties will be considered. 

In [ ]:
# Import the grid for the analysis
filename = "EPN EQ SSH_0401_eq.json"   # Give here the name of the json file with the UKPN grid you want to use
if os.path.isfile(filename):
    net = pp.from_json(filename)
else:
    print("file does not exist, creating a dummy net")
    net = pp.create_empty_network()
    bus = pp.create_bus(net, vn_kv=132)
    pp.create_ext_grid(net, bus=bus)

In [ ]:
# Apply the workarounds on the selected grid
license_area = "EPN"  # Provide here the name of the considered license area. It should be "LPN", "SPN", or "EPN".
if net.bus.index.size > 1:
    remove_impedance = True     # Decide if removing fictious impedances from the grid or not
    net = apply_workarounds(net, license_area, remove_impedance)

Apply the grid reduction to focus the analysis on a limited portion of the overall network

In [ ]:
start_bus = 150    # Select the bus of interest around which you want to reduce the grid

# Call the main function for grid reduction
subnet, trafo_sens_df, trafo3w_sens_df, impedance_sens_df, boundaries, created_ext_grids, created_pq_injections = build_reduced_network(
    net, 
    start_bus=start_bus,            # start bus considered for the reduction
    method="power_flow",            # used method (only power flow available in this tutorial)
    sensitivity_threshold=0.05,     # threshold to decide if cutting or not the subnet
    min_working_current_ka=0.001,   # minimum current limit considered for the cutting
    vn_max_kv=50.0,                 # maximum voltage limit considered for the cutting
    deltaP_MW=1.0,                  # delta of active power toapplied for the sensitivity calculation
    deltaQ_Mvar=0.0,                # delta of reactive power toapplied for the sensitivity calculation
    cut_downward_elements=True)     # decide if apply cuts also in downstream direction (lower voltage levels) or not

if ~np.any(net.bus.index == start_bus):
    print("The selected bus was not found in the considered grid")
else:
    print("kept buses:", len(subnet.bus))
    print("boundaries:", len(boundaries["hv"])+len(boundaries["lv"]))
    print("ext_grids created:", len(created_ext_grids))
    print("pq_injections_created:", len(created_pq_injections))

Run the power flow to create the conditions of the grid assumed as reference values and extract the measurements from such conditions.

In [ ]:
# Run the power flow
pp.runpp(subnet, run_control=False, lightsim2grid=False, max_iteration=100)

# Clean measurements already present in the grid
subnet.measurement.drop(subnet.measurement.index, inplace=True)

# Create the voltage measurements with 0.2% uncertainty
subnet = create_ext_grid_voltage_measurements(subnet, unc=0.2)

# Create the active and reactive power measurements with 1% uncertainty
subnet = create_ext_grid_inj_measurements(subnet, unc=1)

# Remove injection values from shunt or ward elements (which would not be seen as power injection measurements)
remove_shunt_injection_from_meas(subnet,"shunt")
remove_shunt_injection_from_meas(subnet,"ward")

Visualize the measurements that will be used for state estimation purposes

In [ ]:
display(subnet.measurement)
display("Total nodes in the grid: " + str(len(subnet.bus)))
display("Total number of measurements: " + str(len(subnet.measurement)))
display("Measurement redundancy: " + "{:.2f}".format(100*len(subnet.measurement)/(2*len(subnet.bus))) + " %")

Create the load and generation clusters needed for the AF-WLS algorithm

In [ ]:
# Create the load and generation clusters under the load and sgen "type" attribute
subnet.load["type"] = "generic_load"
subnet.sgen["type"] = "generic_sgen"

Modify the nominal values of P and Q for loads and sgens

In [ ]:
# Modify the nominal values of power
subnet.load["p_mw"] *= 1.5
subnet.load["q_mvar"] *= 1.5
subnet.sgen["p_mw"] *= 1.2
subnet.sgen["q_mvar"] *= 1.2

Run state estimation

In [ ]:
# Run the state estimation algorithm
try: 
    success = se.estimate(subnet, algorithm="af-wls", tolerance=1e-4, maximum_iterations=500)
    display("State estimation successfully converged in " + str(success["num_iterations"]) + " iterations.")
except: 
    display("State estimation did not converge")

Compare voltage magnitude state estimation results to reference results given by the initial power flow:

In [ ]:
if subnet.res_bus_est.index.size > 1: 
    volt_diff = subnet.res_bus["vm_pu"].values - subnet.res_bus_est["vm_pu"].values
    max_volt_diff = np.max(abs(volt_diff))
    display("Maximum voltage difference = " + "{:.6f}".format(max_volt_diff) + " p.u.")

Compare line current magnitude state estimation results to reference results given by the initial power flow:

In [ ]:
if subnet.res_line_est.index.size > 1: 
    curr_diff = subnet.res_line["i_from_ka"].values - subnet.res_line_est["i_from_ka"].values
    max_curr_diff = np.max(abs(curr_diff))
    display("Maximum current difference = " + "{:.6f}".format(max_curr_diff) + " kA")

Visualize the estimated allocation factors:

In [ ]:
if hasattr(subnet, "res_cluster_est"):
    display("Estimated load allocation factor = " + "{:.2f}".format(subnet.res_cluster_est[0]))
    display("Estimated generation allocation factor = " + "{:.2f}".format(subnet.res_cluster_est[1]))

### State Forecasting, Example 1 - LPN Grid

State forecasting differs from state estimation because forecast data, instead of real-time measurements, are used to estimate the future operating conditions of the grid. 
The way state forecasting is performed strictly depends on the available forecast data. 

A first option, it to use the time series of real-time measurements for predicting future measurements (same values that are measured - and at the same location - but in the future).
In this case, the same AF-WLS approach seen in the previous examples for state estimation can be used (the only difference is that the values present in the net.measurement table will be forecast values instead of real measurements).

Since forecast data can be generated in multiple ways, also from other data rather than measurement time series, grid operators often have the availability of predicted consumption or generation at the different buses of the grid. 
In this case, a classical Weighted Least Squares algorithm can be used if the forecast data are enough to reach the full observability of the grid. 
The following examples will show the use of the WLS formulation for state forecasting in the case of fully observable grids. 



In [ ]:
# Import the grid for the analysis
filename = "LPN EQ SSH_0401_eq.json"   # Give here the name of the json file with the UKPN grid you want to use
if os.path.isfile(filename):
    net = pp.from_json(filename)
else:
    print("file does not exist, creating a dummy net")
    net = pp.create_empty_network()
    bus = pp.create_bus(net, vn_kv=132)
    pp.create_ext_grid(net, bus=bus)

In [ ]:
# Apply the workarounds on the selected grid
license_area = "LPN"  # Provide here the name of the considered license area. It should be "LPN", "SPN", or "EPN".
if net.bus.index.size > 1:
    remove_impedance = True     # Decide if removing fictious impedances from the grid or not
    net = apply_workarounds(net, license_area, remove_impedance)

Apply the grid reduction to focus the analysis on a limited portion of the overall grid

In [ ]:
start_bus = 1712    # Select the bus of interest around which you want to reduce the grid

# Call the main function for grid reduction
subnet, trafo_sens_df, trafo3w_sens_df, impedance_sens_df, boundaries, created_ext_grids, created_pq_injections = build_reduced_network(
    net, 
    start_bus=start_bus,            # start bus considered for the reduction
    method="power_flow",            # used method (only power flow available in this tutorial)
    sensitivity_threshold=0.05,     # threshold to decide if cutting or not the subnet
    min_working_current_ka=0.001,   # minimum current limit considered for the cutting
    vn_max_kv=50.0,                 # maximum voltage limit considered for the cutting
    deltaP_MW=1.0,                  # delta of active power toapplied for the sensitivity calculation
    deltaQ_Mvar=0.0,                # delta of reactive power toapplied for the sensitivity calculation
    cut_downward_elements=True)     # decide if apply cuts also in downstream direction (lower voltage levels) or not

if ~np.any(net.bus.index == start_bus):
    print("The selected bus was not found in the considered grid")
else:
    print("kept buses:", len(subnet.bus))
    print("boundaries:", len(boundaries["hv"])+len(boundaries["lv"]))
    print("ext_grids created:", len(created_ext_grids))
    print("pq_injections_created:", len(created_pq_injections))

Run a power flow to create the conditions of the grid assumed as reference values and extract forecasted values (with uncertainty) from such conditions.

In [ ]:
# Run the power flow
pp.runpp(subnet, run_control=False, lightsim2grid=False, max_iteration=100)

# Clean measurements already present in the grid
subnet.measurement.drop(subnet.measurement.index, inplace=True)

# Create the voltage measurements with 0.5% uncertainty
subnet = create_ext_grid_voltage_measurements(subnet, unc=0.5)

# Create the active and reactive power measurements with 10% uncertainty
subnet = create_bus_inj_measurements(subnet, unc=10)

# Remove injection values from shunt or ward elements (which would not be seen as power injection measurements)
remove_shunt_injection_from_meas(subnet,"shunt")
remove_shunt_injection_from_meas(subnet,"ward")

Visualize the data that will be used for state forecasting

In [ ]:
display("Total nodes in the grid: " + str(len(subnet.bus)))
display("Total number of forecasts: " + str(len(subnet.measurement)))
display("Data redundancy: " + "{:.2f}".format(100*len(subnet.measurement)/(2*len(subnet.bus))) + " %")

Run the WLS algorithm

In [ ]:
# Run the state forecasting algorithm
try: 
    success = se.estimate(subnet, algorithm="wls", tolerance=1e-6, maximum_iterations=100)
    display("State estimation successfully converged in " + str(success["num_iterations"]) + " iterations.")
except: 
    display("State estimation did not converge")

Compare voltage magnitude state forecasting results to the reference values given by the initial power flow:

In [ ]:
if net.res_bus_est.index.size > 1: 
    volt_diff = subnet.res_bus["vm_pu"].values - subnet.res_bus_est["vm_pu"].values
    max_volt_diff = np.max(abs(volt_diff))
    display("Maximum voltage difference = " + "{:.6f}".format(max_volt_diff) + " p.u.")

Compare line current magnitude state forecasting results to the reference values given by the power flow

In [ ]:
if subnet.res_line_est.index.size > 1: 
    curr_diff = subnet.res_line["i_from_ka"].values - subnet.res_line_est["i_from_ka"].values
    max_curr_diff = np.max(abs(curr_diff))
    display("Maximum current difference = " + "{:.6f}".format(max_curr_diff) + " kA")

### State Forecasting, Example 2 - SPN Grid

In this example, state forecasting will be shown on a portion of the SPN grid.

In [ ]:
# Import the grid for the analysis
filename = "SPN EQ SSH_0401_eq.json"   # Give here the name of the json file with the UKPN grid you want to use
if os.path.isfile(filename):
    net = pp.from_json(filename)
else:
    print("file does not exist, creating a dummy net")
    net = pp.create_empty_network()
    bus = pp.create_bus(net, vn_kv=132)
    pp.create_ext_grid(net, bus=bus)

In [ ]:
# Apply the workarounds on the selected grid
license_area = "SPN"  # Provide here the name of the considered license area. It should be "LPN", "SPN", or "EPN".
if net.bus.index.size > 1:
    remove_impedance = True     # Decide if removing fictious impedances from the grid or not
    net = apply_workarounds(net, license_area, remove_impedance)

Apply the grid reduction to focus the analysis on a limited portion of the overall grid

In [ ]:
start_bus = 4101    # Select the bus of interest around which you want to reduce the grid

# Call the main function for grid reduction
subnet, trafo_sens_df, trafo3w_sens_df, impedance_sens_df, boundaries, created_ext_grids, created_pq_injections = build_reduced_network(
    net, 
    start_bus=start_bus,            # start bus considered for the reduction
    method="power_flow",            # used method (only power flow available in this tutorial)
    sensitivity_threshold=0.05,     # threshold to decide if cutting or not the subnet
    min_working_current_ka=0.001,   # minimum current limit considered for the cutting
    vn_max_kv=50.0,                 # maximum voltage limit considered for the cutting
    deltaP_MW=1.0,                  # delta of active power toapplied for the sensitivity calculation
    deltaQ_Mvar=0.0,                # delta of reactive power toapplied for the sensitivity calculation
    cut_downward_elements=True)     # decide if apply cuts also in downstream direction (lower voltage levels) or not

if ~np.any(net.bus.index == start_bus):
    print("The selected bus was not found in the considered grid")
else:
    print("kept buses:", len(subnet.bus))
    print("boundaries:", len(boundaries["hv"])+len(boundaries["lv"]))
    print("ext_grids created:", len(created_ext_grids))
    print("pq_injections_created:", len(created_pq_injections))

Run a power flow to create the conditions of the grid assumed as reference values and extract forecasted values (with uncertainty) from such conditions.

In [ ]:
# Run the power flow
pp.runpp(subnet, run_control=False, lightsim2grid=False, max_iteration=100)

# Clean measurements already present in the grid
subnet.measurement.drop(subnet.measurement.index, inplace=True)

# Create the voltage measurements with 0.5% uncertainty
subnet = create_ext_grid_voltage_measurements(subnet, unc=0.5)

# Create the active and reactive power measurements with 10% uncertainty
subnet = create_bus_inj_measurements(subnet, unc=10)

# Remove injection values from shunt or ward elements (which would not be seen as power injection measurements)
remove_shunt_injection_from_meas(subnet,"shunt")
remove_shunt_injection_from_meas(subnet,"ward")

Visualize the data that will be used for state forecasting

In [ ]:
display("Total nodes in the grid: " + str(len(subnet.bus)))
display("Total number of forecasts: " + str(len(subnet.measurement)))
display("Data redundancy: " + "{:.2f}".format(100*len(subnet.measurement)/(2*len(subnet.bus))) + " %")

Run the WLS algorithm

In [ ]:
# Run the state forecasting algorithm
try: 
    success = se.estimate(subnet, algorithm="wls", tolerance=1e-6, maximum_iterations=100)
    display("State estimation successfully converged in " + str(success["num_iterations"]) + " iterations.")
except: 
    display("State estimation did not converge")

Compare voltage magnitude state forecasting results to the reference values given by the initial power flow:

In [ ]:
if subnet.res_bus_est.index.size > 1: 
    volt_diff = subnet.res_bus["vm_pu"].values - subnet.res_bus_est["vm_pu"].values
    max_volt_diff = np.max(abs(volt_diff))
    display("Maximum voltage difference = " + "{:.6f}".format(max_volt_diff) + " p.u.")

Compare line current magnitude state forecasting results to the reference values given by the power flow

In [ ]:
if subnet.res_line_est.index.size > 1: 
    curr_diff = subnet.res_line["i_from_ka"].values - subnet.res_line_est["i_from_ka"].values
    max_curr_diff = np.max(abs(curr_diff))
    display("Maximum current difference = " + "{:.6f}".format(max_curr_diff) + " kA")

### State Forecasting, Example 3 - EPN Grid

In this example, state forecasting will be shown on a portion of the EPN grid.

In [ ]:
# Import the grid for the analysis
filename = "EPN EQ SSH_0401_eq.json"   # Give here the name of the json file with the UKPN grid you want to use
if os.path.isfile(filename):
    net = pp.from_json(filename)
else:
    print("file does not exist, creating a dummy net")
    net = pp.create_empty_network()
    bus = pp.create_bus(net, vn_kv=132)
    pp.create_ext_grid(net, bus=bus)

In [ ]:
# Apply the workarounds on the selected grid
license_area = "EPN"  # Provide here the name of the considered license area. It should be "LPN", "SPN", or "EPN".
if net.bus.index.size > 1:
    remove_impedance = True     # Decide if removing fictious impedances from the grid or not
    net = apply_workarounds(net, license_area, remove_impedance)

Apply the grid reduction to focus the analysis on a limited portion of the overall grid

In [ ]:
start_bus = 150    # Select the bus of interest around which you want to reduce the grid

# Call the main function for grid reduction
subnet, trafo_sens_df, trafo3w_sens_df, impedance_sens_df, boundaries, created_ext_grids, created_pq_injections = build_reduced_network(
    net, 
    start_bus=start_bus,            # start bus considered for the reduction
    method="power_flow",            # used method (only power flow available in this tutorial)
    sensitivity_threshold=0.05,     # threshold to decide if cutting or not the subnet
    min_working_current_ka=0.001,   # minimum current limit considered for the cutting
    vn_max_kv=50.0,                 # maximum voltage limit considered for the cutting
    deltaP_MW=1.0,                  # delta of active power toapplied for the sensitivity calculation
    deltaQ_Mvar=0.0,                # delta of reactive power toapplied for the sensitivity calculation
    cut_downward_elements=True)     # decide if apply cuts also in downstream direction (lower voltage levels) or not

if ~np.any(net.bus.index == start_bus):
    print("The selected bus was not found in the considered grid")
else:
    print("kept buses:", len(subnet.bus))
    print("boundaries:", len(boundaries["hv"])+len(boundaries["lv"]))
    print("ext_grids created:", len(created_ext_grids))
    print("pq_injections_created:", len(created_pq_injections))

Run a power flow to create the conditions of the grid assumed as reference values and extract forecasted values (with uncertainty) from such conditions.

In [ ]:
# Run the power flow
pp.runpp(subnet, run_control=False, lightsim2grid=False, max_iteration=100)

# Clean measurements already present in the grid
subnet.measurement.drop(subnet.measurement.index, inplace=True)

# Create the voltage measurements with 0.5% uncertainty
subnet = create_ext_grid_voltage_measurements(subnet, unc=0.5)

# Create the active and reactive power measurements with 10% uncertainty
subnet = create_bus_inj_measurements(subnet, unc=10)

# Remove injection values from shunt or ward elements (which would not be seen as power injection measurements)
remove_shunt_injection_from_meas(subnet,"shunt")
remove_shunt_injection_from_meas(subnet,"ward")

Visualize the data that will be used for state forecasting

In [ ]:
display("Total nodes in the grid: " + str(len(subnet.bus)))
display("Total number of forecasts: " + str(len(subnet.measurement)))
display("Data redundancy: " + "{:.2f}".format(100*len(subnet.measurement)/(2*len(subnet.bus))) + " %")

Run the WLS algorithm

In [ ]:
# Run the state forecasting algorithm
try: 
    success = se.estimate(subnet, algorithm="wls", tolerance=1e-6, maximum_iterations=100)
    display("State estimation successfully converged in " + str(success["num_iterations"]) + " iterations.")
except: 
    display("State estimation did not converge")

Compare voltage magnitude state forecasting results to the reference values given by the initial power flow:

In [ ]:
if subnet.res_bus_est.index.size > 1: 
    volt_diff = subnet.res_bus["vm_pu"].values - subnet.res_bus_est["vm_pu"].values
    max_volt_diff = np.max(abs(volt_diff))
    display("Maximum voltage difference = " + "{:.6f}".format(max_volt_diff) + " p.u.")

Compare line current magnitude state forecasting results to the reference values given by the power flow

In [ ]:
if subnet.res_line_est.index.size > 1: 
    curr_diff = subnet.res_line["i_from_ka"].values - subnet.res_line_est["i_from_ka"].values
    max_curr_diff = np.max(abs(curr_diff))
    display("Maximum current difference = " + "{:.6f}".format(max_curr_diff) + " kA")